In [ ]:
import numpy as np
import xarray as xr
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

mpl.rcParams['axes.unicode_minus'] = False

import cartopy.crs as ccrs


In [ ]:
def ax_pos_inch_to_absolute(fig_size, ax_pos_inch):
    ax_pos_absolute = []
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])

    return ax_pos_absolute

In [ ]:
def ax_pos_cm_to_absolute(fig_size, ax_pos_cm):
    ax_pos_absolute = []
    ax_pos_inch = [ pos / 2.54 for pos in ax_pos_cm ]
    ax_pos_absolute.append(ax_pos_inch[0]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[1]/fig_size[1])
    ax_pos_absolute.append(ax_pos_inch[2]/fig_size[0])
    ax_pos_absolute.append(ax_pos_inch[3]/fig_size[1])
    
    return ax_pos_absolute

In [ ]:
# setting up a custom blue-white-orange color map
from matplotlib.colors import LinearSegmentedColormap

colors = np.array([( 37,  52, 148), ( 44, 127, 184), ( 65, 182, 196), (161, 218, 180), (255, 255, 204),
                   (255, 255, 255),
                   (255, 255, 204), (254, 204,  92), (253, 141,  60), (240,  59,  32), (189,   0,  38)]) / 255.

cmc = LinearSegmentedColormap.from_list('BYWYR', colors, N=101)


In [ ]:
# base dir
base_dir = (Path.cwd() / "../../").resolve()
data_dir = base_dir / "data"
save_dir = base_dir / "figures"

In [ ]:
# raw data
file_name = "subaanual-variability-grid-window-360-skip-180.nc"
ds = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# filtered data
file_name = "olr-2xdaily-1981-2010-window-360-skip-180-filters.nc"
ds_filters = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# filtered background
file_name = "ou-realization-2024-epsilon0-5.8-lambda0-0.06-tau0-2.3-filters.nc"
ds_filters_background = xr.open_dataset(str(data_dir / file_name))

In [ ]:
# extract
londr = ds.long.values
latdr = ds.latg.values
londf = ds_filters.lon.values
latdf = ds_filters.lat.values

In [ ]:
# variance - global mean variance
F = [None] * 4 # list container for all panels
F[0] = ds.std_obs.values**2 - 26.5**2

In [ ]:
# variance - filtered background variance
F[1] = np.mean(np.var(ds_filters.Fw1.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw1, axis=1), axis=0)
F[2] = np.mean(np.var(ds_filters.Fw2.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw2, axis=1), axis=0)
F[3] = np.mean(np.var(ds_filters.Fw3.values, axis=1), axis=0) - np.mean(np.var(ds_filters_background.Fw3, axis=1), axis=0)

In [ ]:
# corrections
for i in range(1, 4):  # for the raw data the correction is done upon processing, no need to do again

    # compensating for hanning window
    F[i] *= (8. / 3.)
    
    # one fourth of the variance is lost when filtering only positive frequencies
    F[i] *= 4


In [ ]:
# clevels (shared for all filters -- saturated at 200)
clevels = [None] * 4

clevels[0] = np.linspace(-1400, 1400, 21)
clevels[1] = np.linspace(-200, 200, 21)
clevels[2] = np.linspace(-200, 200, 21)
clevels[3] = np.linspace(-200, 200, 21)


In [ ]:
fig_size = (07.50/2.54, 17.50/2.54)
fig = plt.figure(figsize=fig_size)

ax = []
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [00.75, 13.25, 06.00, 03.00]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [00.75, 08.00, 06.00, 03.00]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [00.75, 04.50, 06.00, 03.00]), projection=ccrs.PlateCarree(central_longitude=0.0)))
ax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [00.75, 01.00, 06.00, 03.00]), projection=ccrs.PlateCarree(central_longitude=0.0)))

cax = []
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.25, 12.75, 05.00, 00.25])))
cax.append(fig.add_axes(ax_pos_cm_to_absolute(fig_size, [01.25, 00.50, 05.00, 00.25])))


panel_name = ['A', 'B', 'C', 'D']

filter_name = ['Anomalous subannual variability', 
               'Filter 1 (WK99 Kelvin wave)',
               'Filter 2 (Non-disp. eastward branch)',
               'Filter 3 (Non-disp. westward branch)']

for i in range(4):

    if i ==0:
        cs0 = ax[i].contourf(londr, latdr, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))
        cs0 = ax[i].contourf(londr, latdr, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))
    else:
        cs0 = ax[i].contourf(londf, latdf, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))
        cs0 = ax[i].contourf(londf, latdf, F[i], levels=clevels[i], extend='both', cmap=cmc, transform=ccrs.PlateCarree(central_longitude=0.0))

    ax[i].coastlines(linewidths=0.5, color='black')

    ax[i].text(00.01, 00.04, panel_name[i],
               ha='left', va='bottom', transform=ax[i].transAxes, fontsize=10, fontweight='bold', color='black')
    
    ax[i].text(00.50, 01.03, filter_name[i],
               ha='center', va='bottom', transform=ax[i].transAxes, fontsize=10, fontweight='regular', color='black')

    ax[i].gridlines(draw_labels=False, linestyle='--', color='black', alpha=0.2)

# colorbars
    if i == 0:
        
        cbar = fig.colorbar(cs0, cax=cax[i], orientation='horizontal', extend='both')
    
        cax[i].set_xlabel(r'[W m$^{-2}$]$\,^{2}$', fontsize=8, va='top', ha='left', color='black')
        cax[i].xaxis.set_label_coords(01.00, -00.50, transform=cax[i].transAxes)
        cax[i].tick_params(labelsize=8)
    
        cbar.set_ticks([-980, -420, 0, 420, 980])

    elif i == 1:
    
        cbar = fig.colorbar(cs0, cax=cax[i], orientation='horizontal', extend='both')
    
        cax[i].set_xlabel(r'[W m$^{-2}$]$\,^{2}$', fontsize=8, va='top', ha='left', color='black')
        cax[i].xaxis.set_label_coords(01.00, -00.50, transform=cax[i].transAxes)
        cax[i].tick_params(labelsize=8)
    
        cbar.set_ticks([-140, -60, 0, 60, 140])


##### render frames #####

# create a full-canvas background Axis layer
bg_ax = fig.add_axes([0, 0, 1, 1]) 
bg_ax.axis('off') 
bg_ax.set_zorder(-1) # force this entire Axis container to the background

raw_frame = Polygon(
    [(00.02, 00.735), (00.02, 00.97), (00.98, 00.97), (00.98, 00.735)],
    closed=True,
    facecolor="white",
    alpha=1.0,
    edgecolor='grey',
    linewidth=0.5,
    transform=fig.transFigure, 
    figure=fig,
    zorder=0
)

bg_ax.add_patch(raw_frame)

bg_ax.text(00.50, 00.97, 'Outgoing Longwave Radiation',
           ha='center', va='center', transform=bg_ax.transAxes, fontsize=10,
           fontweight='bold', color='black', backgroundcolor='white')


filter_frame = Polygon(
    [(00.02, 00.035), (00.02, 00.68), (00.98, 00.68), (00.98, 00.035)],
    closed=True,
    facecolor="white",
    alpha=1.0,
    edgecolor='grey',
    linewidth=0.5,
    transform=fig.transFigure, 
    figure=fig,
    zorder=0
)

bg_ax.add_patch(filter_frame)

bg_ax.text(00.50, 00.68, 'Spectral filters',
           ha='center', va='center', transform=bg_ax.transAxes, fontsize=10,
           fontweight='bold', color='black', backgroundcolor='white')


In [ ]:
file_name = "fig-01"
Path(save_dir).mkdir(parents=True, exist_ok=True)
fig.savefig(str(save_dir / file_name) + ".png", dpi=600, format='png', facecolor='white')
fig.savefig(str(save_dir / file_name) + ".pdf", dpi=600, format='pdf', facecolor='white')